# Composite Score Test: P-Gated U Ranking

Rank by Û, but **gate** eligibility using P̂ confidence:
- Long candidates: only stocks with P̂ > 0.5 + threshold
- Short candidates: only stocks with P̂ < 0.5 - threshold

Then pick top-k longs and bottom-k shorts from within the gated pools.

If fewer than k stocks pass the gate on a given day, take as many as available.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..') / 'src'))
from krauss.backtest.portfolio import build_daily_portfolios, aggregate_portfolio_returns
from krauss.backtest.costs import compute_turnover, apply_transaction_costs
from krauss.backtest.ranking import rank_and_select

ROOT = Path('..')
PROCESSED = ROOT / 'data' / 'processed'

pred2 = pd.read_parquet(PROCESSED / 'predictions_phase2.parquet')
pred1 = pd.read_parquet(PROCESSED / 'predictions_phase1.parquet')
returns = pd.read_parquet(PROCESSED / 'daily_returns.parquet')

pred2['date'] = pd.to_datetime(pred2['date'])
pred1['date'] = pd.to_datetime(pred1['date'])
returns['date'] = pd.to_datetime(returns['date'])

print(f'Loaded: {len(pred2):,} rows')

Loaded: 2,852,210 rows


In [2]:
def gated_rank_and_select(predictions, p_col, u_col, k, threshold):
    """
    Rank by U, but gate long/short candidates using P confidence.
    
    Long pool:  P > 0.5 + threshold  -> rank by U descending, take top k
    Short pool: P < 0.5 - threshold  -> rank by U ascending, take bottom k
    
    If fewer than k pass the gate, take as many as available.
    """
    df = predictions[['date', 'permno', p_col, u_col]].dropna().copy()
    
    records = []
    for date, day_df in df.groupby('date'):
        # Long candidates: high P confidence, rank by U desc
        long_pool = day_df[day_df[p_col] > 0.5 + threshold]
        if len(long_pool) > 0:
            long_pool = long_pool.nlargest(min(k, len(long_pool)), u_col)
            for _, row in long_pool.iterrows():
                records.append({
                    'date': date, 'permno': row['permno'],
                    'rank': 0, 'side': 'long', 'score': row[u_col],
                })
        
        # Short candidates: low P confidence, rank by U asc
        short_pool = day_df[day_df[p_col] < 0.5 - threshold]
        if len(short_pool) > 0:
            short_pool = short_pool.nsmallest(min(k, len(short_pool)), u_col)
            for _, row in short_pool.iterrows():
                records.append({
                    'date': date, 'permno': row['permno'],
                    'rank': 0, 'side': 'short', 'score': row[u_col],
                })
    
    return pd.DataFrame(records)


def run_gated_backtest(predictions, p_col, u_col, k, threshold, returns_df, cost_bps=5):
    """Run backtest with P-gated U ranking."""
    sel = gated_rank_and_select(predictions, p_col, u_col, k, threshold)
    if len(sel) == 0:
        return None
    
    # build_daily_portfolios expects rank_and_select output format
    # We need to handle variable k per day since gating may reduce counts
    ret = returns_df[['permno', 'date', 'ret']].sort_values(['permno', 'date'])
    ret['next_date'] = ret.groupby('permno')['date'].shift(-1)
    ret['next_day_ret'] = ret.groupby('permno')['ret'].shift(-1)
    ret = ret.dropna(subset=['next_day_ret'])
    ret['next_date'] = ret['next_date'].astype('datetime64[ns]')
    
    holdings = sel.merge(
        ret[['permno', 'date', 'next_date', 'next_day_ret']],
        on=['date', 'permno'], how='inner',
    )
    
    # Equal weight within each side per day (variable k)
    for side in ['long', 'short']:
        mask = holdings['side'] == side
        counts = holdings[mask].groupby('date')['permno'].transform('count')
        sign = 1.0 if side == 'long' else -1.0
        holdings.loc[mask, 'weight'] = sign / counts
    
    holdings['contrib'] = holdings['weight'] * holdings['next_day_ret']
    
    # Aggregate
    long = holdings[holdings['side'] == 'long']
    short = holdings[holdings['side'] == 'short']
    
    long_agg = long.groupby('date').agg(
        long_ret=('next_day_ret', 'mean'),
        n_long=('permno', 'count'),
        next_date=('next_date', 'first'),
    )
    short_agg = short.groupby('date').agg(
        short_ret=('next_day_ret', 'mean'),
        n_short=('permno', 'count'),
    )
    
    daily = long_agg.join(short_agg, how='inner')
    daily['port_ret'] = daily['long_ret'] - daily['short_ret']
    daily = daily.reset_index()
    daily['date'] = pd.to_datetime(daily['date'])
    
    # Turnover and costs
    turn = compute_turnover(holdings, k=k)
    daily = apply_transaction_costs(daily, turn, cost_bps)
    
    return daily


def run_backtest(predictions, score_col, k, returns_df, cost_bps=5):
    """Standard backtest (no gating)."""
    sel = rank_and_select(predictions, k=k, score_col=score_col)
    hold = build_daily_portfolios(sel, returns_df, k=k)
    daily = aggregate_portfolio_returns(hold)
    turn = compute_turnover(hold, k=k)
    daily = apply_transaction_costs(daily, turn, cost_bps)
    daily['date'] = pd.to_datetime(daily['date'])
    return daily

print('Functions defined.')

Functions defined.


In [8]:
# Check how many stocks pass the gate at various thresholds per model
import warnings
warnings.filterwarnings('ignore')

thresholds = [0.00, 0.01, 0.02, 0.03, 0.05, 0.07, 0.10]
model_names = {'dnn': 'DNN', 'xgb': 'XGB', 'rf': 'RF', 'ens1': 'ENS1'}

gate_rows = []
n_total = pred2.groupby('date')['permno'].count().mean()

for family, m_name in model_names.items():
    p_col = f'p_{family}'
    for thresh in thresholds:
        n_long = (pred2[p_col] > 0.5 + thresh).groupby(pred2['date']).sum().mean()
        n_short = (pred2[p_col] < 0.5 - thresh).groupby(pred2['date']).sum().mean()
        gate_rows.append({
            'Model': m_name, 'Threshold': thresh,
            'Long': int(round(n_long)), 'Short': int(round(n_short)),
            'Long %': n_long / n_total * 100, 'Short %': n_short / n_total * 100,
        })

gate_df = pd.DataFrame(gate_rows).set_index(['Model', 'Threshold'])
display(gate_df.style.format({
    'Long': '{:d}', 'Short': '{:d}',
    'Long %': '{:.1f}', 'Short %': '{:.1f}',
}).set_caption(f'Avg daily stocks passing P-gate (universe ~{int(round(n_total))})'))

In [4]:
K_VALUES = [10, 50]
THRESHOLDS = [0.02, 0.03, 0.05]

results = []

for family, p1_col in [('dnn', 'p_dnn'), ('xgb', 'p_xgb'), ('rf', 'p_rf'), ('ens1', 'p_ens1')]:
    m_name = family.upper()
    for k in K_VALUES:
        # P1 baseline
        d = run_backtest(pred1, p1_col, k, returns)
        results.append({'Model': m_name, 'Score': 'P1 Base', 'k': k,
                        'ret/day': d['port_ret'].mean(), 'ret_net/day': d['port_ret_net'].mean(),
                        'n_days': len(d)})
        
        # P2 U-only (ungated)
        d = run_backtest(pred2, f'score_u_{family}', k, returns)
        results.append({'Model': m_name, 'Score': 'P2 U (no gate)', 'k': k,
                        'ret/day': d['port_ret'].mean(), 'ret_net/day': d['port_ret_net'].mean(),
                        'n_days': len(d)})
        
        # Gated variants
        for thresh in THRESHOLDS:
            d = run_gated_backtest(pred2, f'p_{family}', f'u_{family}', k, thresh, returns)
            if d is not None and len(d) > 0:
                results.append({
                    'Model': m_name, 'Score': f'P-gate({thresh:.2f}) + U',
                    'k': k,
                    'ret/day': d['port_ret'].mean(),
                    'ret_net/day': d['port_ret_net'].mean(),
                    'n_days': len(d),
                })
        
        print(f'  {m_name} k={k} done')

res = pd.DataFrame(results)
print('\nAll done.')

  DNN k=10 done
  DNN k=50 done
  XGB k=10 done
  XGB k=50 done
  RF k=10 done
  RF k=50 done
  ENS1 k=10 done
  ENS1 k=50 done

All done.


In [5]:
# k=10 headline across all models
score_order = ['P1 Base', 'P2 U (no gate)'] + [f'P-gate({t:.2f}) + U' for t in THRESHOLDS]

k10 = res[res['k'] == 10].pivot(index='Model', columns='Score', values='ret_net/day')
k10 = k10.reindex(columns=[c for c in score_order if c in k10.columns])
k10 = (k10 * 100).round(4)
k10.columns.name = 'Daily net return (%)'
display(k10.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=1)
        .set_caption('k=10 post-cost daily return (%)'))

print()

# ENS1 across all k
ens1 = res[res['Model'] == 'ENS1'].pivot(index='Score', columns='k', values='ret_net/day')
ens1 = ens1.reindex([c for c in score_order if c in ens1.index])
ens1 = (ens1 * 100).round(4)
display(ens1.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=None)
        .set_caption('ENS1 post-cost daily return (%) across k'))

print()

# Coverage: how many trading days have both sides populated?
ens1_days = res[(res['Model'] == 'ENS1') & (res['k'] == 10)][['Score', 'n_days']].set_index('Score')
display(ens1_days.style.set_caption('ENS1 k=10: trading days with both long and short positions'))

Daily net return (%),P1 Base,P2 U (no gate),P-gate(0.02) + U,P-gate(0.03) + U,P-gate(0.05) + U
Model,,,,,
DNN,0.1370,0.0822,0.1869,0.5720,6.1162
ENS1,0.2788,0.2567,0.2621,0.3475,0.7193
RF,0.2709,0.2200,0.2536,0.2463,0.3151
XGB,0.2149,0.1835,0.1840,0.2038,0.3166


k,10,50
Score,,
P1 Base,0.2788,0.1208
P2 U (no gate),0.2567,0.1012
P-gate(0.02) + U,0.2621,0.1547
P-gate(0.03) + U,0.3475,0.2556
P-gate(0.05) + U,0.7193,0.6295


,n_days
Score,
P1 Base,5750
P2 U (no gate),5750
P-gate(0.02) + U,5667
P-gate(0.03) + U,5142
P-gate(0.05) + U,2703


In [ ]:
# Trading coverage stats for DNN and ENS1 at gate thresholds 0.03 and 0.05
coverage_rows = []
total_days = pred2['date'].nunique()

for family, m_name in [('dnn', 'DNN'), ('ens1', 'ENS1')]:
    p_col = f'p_{family}'
    for thresh in [0.03, 0.05]:
        long_flags = (pred2[p_col] > 0.5 + thresh).groupby(pred2['date']).sum()
        short_flags = (pred2[p_col] < 0.5 - thresh).groupby(pred2['date']).sum()

        has_long = long_flags > 0
        has_short = short_flags > 0

        both = (has_long & has_short).sum()
        long_only = (has_long & ~has_short).sum()
        short_only = (~has_long & has_short).sum()
        neither = (~has_long & ~has_short).sum()

        coverage_rows.append({
            'Model': m_name,
            'Threshold': thresh,
            'Total days': total_days,
            'Both sides': int(both),
            'Long only': int(long_only),
            'Short only': int(short_only),
            'Neither': int(neither),
            'Avg long candidates': long_flags[has_long].mean() if has_long.any() else 0,
            'Avg short candidates': short_flags[has_short].mean() if has_short.any() else 0,
        })

cov = pd.DataFrame(coverage_rows).set_index(['Model', 'Threshold'])
display(cov.style.format({
    'Total days': '{:,}',
    'Both sides': '{:,}',
    'Long only': '{:,}',
    'Short only': '{:,}',
    'Neither': '{:,}',
    'Avg long candidates': '{:.1f}',
    'Avg short candidates': '{:.1f}',
}).set_caption('Trading day coverage: DNN & ENS1 at gate 0.03 and 0.05'))

In [11]:
from datasets import load_dataset

ds = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="likes.parquet")
